# MATH840 — Lab 2: Reading a series, and the bar it sets

**Week 2 | graded assignment 1 of 7 | 8 points | due at the end of the session**

⚠️ **This lab is on last week's lecture, not today's.** From now on the practice runs one
week behind the lecture, so missing a lecture no longer costs you the lab that follows it.
Everything you need today is in the [Week 1 deck](https://01a05924-a704-8747-1dda-9917ad1a3d63.share.connect.posit.cloud).

Today: look at your series properly, commit to a claim about which simple method will be
hardest to beat on it, then find out.

| Section | Criteria | Points |
|---|---|---|
| 1 | Data preparation | 1 |
| 2 | EDA and visualisation | 4 |
| 3 | Implementation | 2 |
| 4 | Code quality and reproducibility | 1 |

Submit `SURNAME_lab02.pdf`, `SURNAME_lab02.ipynb` and `SURNAME_lab02_summary.csv` to Moodle
before the end of the session.

## 0. Setup

Given — run it and move on.

In [ ]:
!pip install -q statsforecast utilsforecast

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

print("pandas", pd.__version__)

In [ ]:
SURNAME = ""   # <-- for your submission file names
MY_ID   = ""   # <-- the SAME identifier you used in Week 1

if not SURNAME.strip() or not MY_ID.strip():
    raise ValueError("Set SURNAME and MY_ID before running the rest.")

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")

POOL = (
    [{"file": "aus_production.csv", "column": c, "freq": "QS", "season": 4,
      "label": f"Australian production: {c}"}
     for c in ["Beer", "Tobacco", "Bricks", "Cement", "Electricity", "Gas"]]
    + [{"file": "tourism.csv", "state": s, "purpose": p, "freq": "QS", "season": 4,
        "label": f"Tourism: {p} trips to {s}"}
       for s in ["South Australia", "Northern Territory", "Western Australia",
                 "Victoria", "New South Wales", "Queensland", "ACT", "Tasmania"]
       for p in ["Business", "Holiday", "Other", "Visiting"]]
    + [{"file": "ansett.csv", "route": r, "class": c, "freq": "W-MON", "season": 52,
        "label": f"Ansett passengers: {r}, {c} class"}
       for r in ["MEL-SYD", "MEL-ADL", "SYD-BNE", "MEL-BNE", "ADL-PER", "MEL-PER"]
       for c in ["Business", "Economy"]]
)


def assign_dataset(student_id: str) -> dict:
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return POOL[int(digest, 16) % len(POOL)]


def load_my_series(spec: dict) -> pd.DataFrame:
    df = pd.read_csv(f"{BASE}/{spec['file']}")
    if spec["file"] == "aus_production.csv":
        out = df[["ds", spec["column"]]].rename(columns={spec["column"]: "y"})
    elif spec["file"] == "tourism.csv":
        sub = df[(df["State"] == spec["state"]) & (df["Purpose"] == spec["purpose"])]
        out = sub.groupby("ds", as_index=False)["y"].sum()
    else:
        sub = df[(df["Airports"] == spec["route"]) & (df["Class"] == spec["class"])]
        out = sub[["ds", "y"]].copy()
    out["ds"] = pd.to_datetime(out["ds"])
    return out.sort_values("ds").reset_index(drop=True)


mine = assign_dataset(MY_ID)
raw = load_my_series(mine)

SEASON = mine["season"]
H = SEASON if SEASON >= 12 else 2 * SEASON   # holdout: one year of data

print(mine["label"], "| freq", mine["freq"], "| season", SEASON, "| holdout", H)
raw.head()

## 1. Data preparation

*1 point.* Resolve duplicates, put the series on a regular grid, and state the four facts:
frequency, span, complete cycles, and what you did about duplicates and gaps.

**⚠️ Where this goes wrong: the frequency string.** `"W"` means *week ending Sunday*, so a
Monday series reindexed with it comes back empty — use `spec["freq"]`, which is already the
anchored form. pandas 3 removed the bare `"M"`, `"Q"`, `"Y"` and `"H"` aliases. After
building the grid, check `s["y"].notna().sum()` is close to what you started with.

📖 [offset aliases](https://pandas.pydata.org/docs/user_guide/timeseries.html#offset-aliases) ·
[anchored offsets](https://pandas.pydata.org/docs/user_guide/timeseries.html#anchored-offsets) ·
[`reindex`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reindex.html) ·
[`duplicated`](https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html)

**⚠️ If your series ends with missing values.** Some series stop being reported years before
their last row. `interpolate()` fills that tail by repeating the last value, and then your
holdout is a flat line you invented — every method predicts it perfectly and the exercise is
meaningless. **Trim to the last real observation** before splitting, and say so in your
report. Filling interior gaps is repair; filling the tail is fabrication.



In [ ]:
# TODO: duplicates, regular grid, the four facts.
# End with a clean frame called `s` with columns ds, y.

s = raw.copy()

**Your description.** *(frequency, span, complete cycles, duplicates, gaps)*

→

## 2. EDA and visualisation

*4 points.* Four plots, each with a reading, and then a claim.

**⚠️ Where this goes wrong.** `plot_acf` raises once `lags` reaches the length of the
series — `lags=min(40, len(y) // 3)` is safe, but use at least 60 for weekly data. Remove
missing values first; `plot_acf` will not handle `NaN`.

📖 [`plot_acf`](https://www.statsmodels.org/stable/generated/statsmodels.graphics.tsaplots.plot_acf.html)

In [ ]:
# TODO: time plot

In [ ]:
# TODO: seasonal plot

In [ ]:
# TODO: seasonal subseries plot

In [ ]:
# TODO: ACF

**What you see.** Name the patterns: trend, seasonality, cycles, level shifts, outliers.
Say whether the seasonal swing grows with the level — you will need that next week.

→

### Your prediction

Which of the four methods will be **hardest to beat** on your series?

Argue from the plots: the size of the seasonal swing against the size of the trend, whether
the pattern repeats in the same shape, whether the level moved recently.

⚠️ Write this **before** running Section 3. A well-argued wrong prediction earns full
marks; an unargued right one earns nothing.

In [ ]:
PREDICTION = ""   # one of: "mean", "naive", "drift", "snaive"

assert PREDICTION in {"mean", "naive", "drift", "snaive"}, "pick one of the four"

**Your argument.**

→

## 3. Implementation

*2 points.* Implement the four methods, hold out the last year, and score them.

$$
\hat{y}_{T+h} = \bar{y}
\qquad
\hat{y}_{T+h} = y_T
\qquad
\hat{y}_{T+h} = y_{T+h-m}
\qquad
\hat{y}_{T+h} = y_T + h\,\frac{y_T - y_1}{T-1}
$$

**⚠️ Where this goes wrong.** `snaive` must use the last `m` values of the **training**
part — if your forecast looks perfect, you are forecasting with the answer. Handle missing
values before scoring, or every RMSE comes out `NaN`.

In [ ]:
def benchmark_forecasts(y: np.ndarray, h: int, m: int) -> dict:
    """mean, naive, snaive and drift, h steps ahead."""
    # TODO
    raise NotImplementedError

In [ ]:
# TODO: split off the last H observations as the holdout, forecast them with all four
# methods, and compute the RMSE of each.

results = None   # a DataFrame with one row per method: rmse, and ratio to the best
results

**The verdict.** Which method won, and by how much? Did your prediction hold? If it did
not, say what in the plots you misread — that sentence is worth more than the prediction
was.

→

## 4. Code quality and reproducibility

*1 point.* Restart the kernel, run everything top to bottom, delete dead cells, and check
that every number in your text comes from a cell.

## 5. Submit

Run the cell below, then export and upload **three files** to the Week 2 activity on
Moodle before the end of the session:

- `SURNAME_lab02.pdf` — `File → Print → Save as PDF`
- `SURNAME_lab02.ipynb` — `File → Download → Download .ipynb`
- `SURNAME_lab02_summary.csv` — written below

In [ ]:
winner = results["rmse"].idxmin() if results is not None else None

summary = pd.DataFrame([{
    "student_id": MY_ID,
    "series": mine["label"],
    "prediction": PREDICTION,
    "winner": winner,
    "correct": PREDICTION == winner,
    **{f"rmse_{k}": round(float(v), 3) for k, v in results["rmse"].items()},
}])

name = f"{SURNAME.strip().upper()}_lab02_summary.csv"
summary.to_csv(name, index=False)
print(summary.to_string(index=False))

try:
    from google.colab import files
    files.download(name)
except ImportError:
    print(f"\nNot in Colab — {name} is saved next to this notebook.")